# Issue #20: Movie Popularity Prediction with Gradient Boosting
Build a Gradient Boosting regression model to predict movie popularity using the feature-engineered popularity dataset, `09_popularity_features.csv`, with the target being `target_log_popularity`.

## 0. Setup

In [ ]:
import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET = "target_log_popularity"

EXCLUDED_COLUMNS = {
    "id",
    TARGET,
    "log_revenue",
    "revenue_reported",
    "title",
    "original_title",
    "overview",
    "genres",
    "production_companies",
    "production_countries",
    "spoken_languages",
    "director",
    "top_cast",
    "keywords",
}

CATEGORICAL_FEATURES = ["original_language", "status"]

## 1. ?

In [ ]:
def load_and_prepare_data(data_path: Path) -> tuple[pd.DataFrame, pd.Series, list[str], list[str]]:
    """Load the CSV and return X, y, numeric columns, and categorical columns."""
    df = pd.read_csv(data_path)
    missing = {TARGET} - set(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s): {sorted(missing)}")

    usable = [column for column in df.columns if column not in EXCLUDED_COLUMNS]
    X = df[usable].copy()
    y = pd.to_numeric(df[TARGET], errors="coerce")

    # Convert accidental numeric strings to numeric values where possible.
    for column in X.columns:
        if column not in CATEGORICAL_FEATURES:
            converted = pd.to_numeric(X[column], errors="coerce")
            if converted.notna().mean() >= 0.90:
                X[column] = converted

    valid_rows = y.notna()
    X, y = X.loc[valid_rows].reset_index(drop=True), y.loc[valid_rows].reset_index(drop=True)
    categorical = [c for c in CATEGORICAL_FEATURES if c in X.columns]
    numeric = [c for c in X.columns if c not in categorical]
    return X, y, numeric, categorical

## 2. ?

In [ ]:
def make_preprocessor(numeric: list[str], categorical: list[str]) -> ColumnTransformer:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric),
        ("categorical", categorical_pipeline, categorical),
    ], remainder="drop")

## 3. ?

In [ ]:
def metrics(actual: pd.Series | np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }

## 4. ?

In [ ]:
def save_validation_plot(results: pd.DataFrame, output_path: Path) -> None:
    labels = [f"M{i + 1}" for i in range(len(results))]
    x = np.arange(len(labels))
    width = 0.25
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, metric_name, color in zip(
        axes, ["MAE", "RMSE", "R2"], ["#4472C4", "#ED7D31", "#70AD47"]
    ):
        axis.bar(x, results[metric_name], color=color)
        axis.set_title(f"Validation {metric_name}")
        axis.set_xlabel("Model configuration")
        axis.set_ylabel(metric_name)
        axis.set_xticks(x, labels)
        axis.grid(axis="y", alpha=0.25)
    fig.suptitle("Validation performance across Gradient Boosting configurations")
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

def save_predicted_plot(actual: pd.Series, predicted: np.ndarray, output_path: Path) -> None:
    low = min(float(actual.min()), float(predicted.min()))
    high = max(float(actual.max()), float(predicted.max()))
    fig, axis = plt.subplots(figsize=(7, 6))
    axis.scatter(actual, predicted, alpha=0.65, edgecolor="none")
    axis.plot([low, high], [low, high], "r--", label="Perfect prediction")
    axis.set_title("Test Set: Predicted vs. Actual Log Popularity")
    axis.set_xlabel("Actual target_log_popularity")
    axis.set_ylabel("Predicted target_log_popularity")
    axis.legend()
    axis.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

## 5. ?

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data",
        type=Path,
        default=Path("data") / "features" / "09_popularity_features.csv",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("data") / "modeling",
    )
    args = parser.parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)

    X, y, numeric, categorical = load_and_prepare_data(args.data)
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

    configurations = [
        {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2},
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2},
        {"n_estimators": 100, "learning_rate": 0.10, "max_depth": 2},
        {"n_estimators": 200, "learning_rate": 0.10, "max_depth": 2},
        {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 3},
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3},
        {"n_estimators": 100, "learning_rate": 0.10, "max_depth": 3},
        {"n_estimators": 200, "learning_rate": 0.10, "max_depth": 3},
    ]

    validation_rows = []
    for config in configurations:
        model = Pipeline([
            ("preprocess", make_preprocessor(numeric, categorical)),
            ("regressor", GradientBoostingRegressor(random_state=42, **config)),
        ])
        model.fit(X_train, y_train)
        validation_rows.append({**config, **metrics(y_val, model.predict(X_val))})

    validation_results = pd.DataFrame(validation_rows)
    best_index = validation_results["RMSE"].idxmin()
    best_config = {key: validation_results.loc[best_index, key] for key in configurations[0]}
    best_config = {key: int(value) if key != "learning_rate" else float(value)
                   for key, value in best_config.items()}

    selected_model = Pipeline([
        ("preprocess", make_preprocessor(numeric, categorical)),
        ("regressor", GradientBoostingRegressor(random_state=42, **best_config)),
    ])
    
    X_train_val = pd.concat([X_train, X_val])
    y_train_val = pd.concat([y_train, y_val])
    selected_model.fit(X_train_val, y_train_val)
    test_predictions = selected_model.predict(X_test)
    test_metrics = metrics(y_test, test_predictions)

    validation_results.to_csv(args.output_dir / "validation_results.csv", index=False)
    save_validation_plot(validation_results, args.output_dir / "validation_model_comparison.png")
    save_predicted_plot(y_test, test_predictions, args.output_dir / "test_predicted_vs_actual.png")

    sample = X_test.iloc[[0]]
    sample_prediction = float(selected_model.predict(sample)[0])
    print("Features used:", list(X.columns))
    print(f"Split sizes: train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}")
    print("\nValidation results:")
    print(validation_results.to_string(index=False))
    print("\nSelected configuration (lowest validation RMSE):", best_config)
    print("Final test metrics:", {k: round(v, 4) for k, v in test_metrics.items()})
    print(f"\nInference example: predicted target_log_popularity = {sample_prediction:.4f}")

if __name__ == "__main__":
    main()

## Results

Creates `data\modeling` folder and 3 output files, `test_predicted_vs_actual.png`, `validation_model_comparison.png`, and `validation_results.csv`.   

Terminal output:  
Validation results:  
| n_estimators | learning_rate | max_depth | MAE | RMSE | R² |
|----------|----------|----------|----------|----------|----------|
| 100 | 0.05 | 2 | 0.420900 | 0.537301 | 0.545987 |
| 200 | 0.05 | 2 | 0.404541 | 0.522061 | 0.571377 |
| 100 | 0.10 | 2 | 0.404120 | 0.522254 | 0.571060 |
| 200 | 0.10 | 2 | 0.397683 | 0.515366 | 0.582301 |
| 100 | 0.05 | 3 | 0.405573 | 0.523286 | 0.569365 |
| 200 | 0.05 | 3 | 0.394614 | 0.512611 | 0.586755 |
| 100 | 0.10 | 3 | 0.394878 | 0.512889 | 0.586307 |
| 200 | 0.10 | 3 | 0.390236 | 0.507570 | 0.594842 |

Selected configuration (lowest validation RMSE): {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 3}  
Final test metrics: {'MAE': 0.3856, 'RMSE': 0.5015, 'R²': 0.6136}  

Inference example: predicted target_log_popularity = 0.6103  